# NiyamTrace-X — Three-API Paper Closure Notebook

This notebook replaces the OpenRouter-dependent closure path.

At runtime it securely asks for:

1. **Gemini API key**
2. **OpenAI API key**
3. **Groq API key**

The keys are entered with `getpass()` and are **never written into result files, manifests, logs, or ZIP archives**.

## Three independent model families

The preferred closure matrix is:

| Family | Preferred provider/model | Automatic fallback |
|---|---|---|
| OpenAI | OpenAI `gpt-5.6-luna` | Groq `openai/gpt-oss-20b` |
| Gemini | Gemini `gemini-2.5-flash-lite` | Gemini `gemini-2.5-flash` |
| Qwen | Groq `qwen/qwen3.6-27b` | Groq `qwen/qwen3.8-27b` |

So even if the OpenAI API key has no usable credits, the notebook can still preserve an independent OpenAI/GPT-OSS family through Groq.

## Required external closure benchmarks

- **BFCL-v4**
- **AgentDojo**
- **τ³ / tau2-bench**

The notebook only marks a benchmark slice `SUPPORTED` when native evaluated outputs exist.

## Final files

- `NTX_3API_RAW_DATA.zip`
- `NTX_3API_PROCESSED_RESULTS.zip`
- `NTX_3API_PAPER_INTEGRATION.zip`
- `NTX_3API_PAPER_CLOSURE_MASTER.zip`

Send the **master ZIP** back after the run.

In [ ]:
# ============================================================
# CELL 1 — GLOBAL CONFIGURATION
# ============================================================
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone
import os, sys, json, re, random, hashlib, shutil, zipfile, subprocess, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)

SELFTEST=os.getenv("NTX_SELFTEST","0")=="1"
MODE=os.getenv("NTX_CLOSURE_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE","FULL"}

BASE=Path(os.getenv(
    "NTX_3API_DIR",
    "/content/NTX_3API_CLOSURE" if Path("/content").exists() else str(Path.cwd()/"NTX_3API_CLOSURE")
))
WORK=BASE/"work"
RAW=BASE/"raw"
RESULTS=BASE/"results"
LOGS=BASE/"logs"
ENV=BASE/"environment"
PAPER=BASE/"paper_integration"
ARCH=BASE/"archives"

for p in [BASE,WORK,RAW,RESULTS,LOGS,ENV,PAPER,ARCH]:
    p.mkdir(parents=True,exist_ok=True)

CFG={
"SMOKE":{
    "bfcl_limit":5,
    "dojo_suites":["banking"],
    "dojo_user_tasks":["user_task_0"],
    "dojo_injection_tasks":["injection_task_0"],
    "tau_domains":["airline","retail","telecom"],
    "tau_num_tasks":1,
    "tau_max_steps":18,
    "agent_max_tokens":768,
    "user_max_tokens":384,
    "bootstrap":500,
},
"CLOSURE":{
    "bfcl_limit":100,
    "dojo_suites":["banking","workspace","travel","slack"],
    "dojo_user_tasks":["user_task_0","user_task_5","user_task_10"],
    "dojo_injection_tasks":["injection_task_0","injection_task_1"],
    "tau_domains":["airline","retail","telecom"],
    "tau_num_tasks":10,
    "tau_max_steps":50,
    "agent_max_tokens":1536,
    "user_max_tokens":768,
    "bootstrap":5000,
},
"FULL":{
    "bfcl_limit":None,
    "dojo_suites":["banking","workspace","travel","slack"],
    "dojo_user_tasks":None,
    "dojo_injection_tasks":None,
    "tau_domains":["airline","retail","telecom"],
    "tau_num_tasks":None,
    "tau_max_steps":100,
    "agent_max_tokens":2048,
    "user_max_tokens":1024,
    "bootstrap":10000,
},
}[MODE]

PREVIOUS_RESULTS_ZIP=os.getenv("NTX_PREVIOUS_RESULTS_ZIP","").strip()

print("MODE:",MODE)
print("SELFTEST:",SELFTEST)
print("BASE:",BASE)
print(json.dumps(CFG,indent=2))

In [ ]:
# ============================================================
# CELL 2 — HELPERS / CHECKPOINTS / ARCHIVES
# ============================================================
CHECKPOINT=RESULTS/"checkpoint.json"

def now():
    return datetime.now(timezone.utc).isoformat()

def run(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
    )

def save_log(name,p,cmd=None):
    parts=[]
    if cmd is not None:
        parts += ["COMMAND"," ".join(map(str,cmd)),""]
    parts += ["STDOUT",p.stdout or "","","STDERR",p.stderr or ""]
    (LOGS/name).write_text("\n".join(parts),errors="ignore")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def save_json(path,obj):
    Path(path).write_text(json.dumps(obj,indent=2,default=str))

def load_json(path,default=None):
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return {} if default is None else default

STATE=load_json(CHECKPOINT,{})
def ck_get(key): return STATE.get(key,{})
def ck_set(key,status,**extra):
    STATE[key]={"status":status,"updated_at":now(),**extra}
    save_json(CHECKPOINT,STATE)
def supported(key): return ck_get(key).get("status")=="SUPPORTED"

def ensure_uv():
    if not shutil.which("uv"):
        p=run([sys.executable,"-m","pip","install","-q","-U","uv"])
        save_log("uv_install.log",p)
        if p.returncode:
            raise RuntimeError("uv installation failed")

def clone(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=run(["git","clone","--depth","1",url,dest])
        save_log(f"clone_{dest.name}.log",p)
        if p.returncode:
            raise RuntimeError(f"clone failed: {url}")
    commit=run(["git","-C",dest,"rev-parse","HEAD"]).stdout.strip()
    status=run(["git","-C",dest,"status","--porcelain"]).stdout
    return commit,status

def copytree(src,dst):
    src,dst=Path(src),Path(dst)
    if not src.exists():
        return False
    shutil.rmtree(dst,ignore_errors=True)
    shutil.copytree(src,dst)
    return True

def make_zip(src,dst):
    src,dst=Path(src),Path(dst)
    if dst.exists():
        dst.unlink()
    with zipfile.ZipFile(dst,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for p in sorted(src.rglob("*")):
            if p.is_file():
                z.write(p,arcname=str(p.relative_to(src)))
    return dst

print("Checkpoint entries:",len(STATE))

## Enter the three API keys

This cell asks for all three keys using hidden prompts.

If an OpenAI API key exists but has no API billing/credits, the OpenAI family automatically falls back to **GPT-OSS on Groq**.

Do not hard-code keys into this notebook.

In [ ]:
# ============================================================
# CELL 3 — SECURE API KEY PROMPTS
# ============================================================
if SELFTEST:
    GEMINI_API_KEY="SELFTEST_GEMINI"
    OPENAI_API_KEY="SELFTEST_OPENAI"
    GROQ_API_KEY="SELFTEST_GROQ"
else:
    GEMINI_API_KEY=os.getenv("GEMINI_API_KEY","").strip()
    OPENAI_API_KEY=os.getenv("OPENAI_API_KEY","").strip()
    GROQ_API_KEY=os.getenv("GROQ_API_KEY","").strip()

    if not GEMINI_API_KEY:
        GEMINI_API_KEY=getpass("Gemini API key (hidden): ").strip()

    if not OPENAI_API_KEY:
        OPENAI_API_KEY=getpass(
            "OpenAI API key (hidden; press Enter if unavailable): "
        ).strip()

    if not GROQ_API_KEY:
        GROQ_API_KEY=getpass("Groq API key (hidden): ").strip()

    if not GEMINI_API_KEY:
        raise RuntimeError("Gemini API key is required.")

    if not GROQ_API_KEY:
        raise RuntimeError("Groq API key is required.")

print("Gemini key:", "SET" if GEMINI_API_KEY else "MISSING")
print("OpenAI key:", "SET" if OPENAI_API_KEY else "MISSING/OPTIONAL")
print("Groq key:", "SET" if GROQ_API_KEY else "MISSING")

In [ ]:
# ============================================================
# CELL 4 — PROVIDER / MODEL MATRIX
# ============================================================
OPENAI_BASE="https://api.openai.com/v1"
GEMINI_BASE="https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE="https://api.groq.com/openai/v1"

# Each candidate is a complete provider route.
FAMILY_ROUTES=[
    {
        "family":"OpenAI",
        "label":"openai",
        "routes":[
            {
                "provider":"openai",
                "base_url":OPENAI_BASE,
                "api_key_name":"OPENAI_API_KEY",
                "model":"gpt-5.6-luna",
                "tau_model":"openai/gpt-5.6-luna",
            },
            {
                "provider":"groq",
                "base_url":GROQ_BASE,
                "api_key_name":"GROQ_API_KEY",
                "model":"openai/gpt-oss-20b",
                "tau_model":"groq/openai/gpt-oss-20b",
            },
            {
                "provider":"groq",
                "base_url":GROQ_BASE,
                "api_key_name":"GROQ_API_KEY",
                "model":"openai/gpt-oss-120b",
                "tau_model":"groq/openai/gpt-oss-120b",
            },
        ],
    },
    {
        "family":"Gemini",
        "label":"gemini",
        "routes":[
            {
                "provider":"gemini",
                "base_url":GEMINI_BASE,
                "api_key_name":"GEMINI_API_KEY",
                "model":"gemini-2.5-flash-lite",
                "tau_model":"gemini/gemini-2.5-flash-lite",
            },
            {
                "provider":"gemini",
                "base_url":GEMINI_BASE,
                "api_key_name":"GEMINI_API_KEY",
                "model":"gemini-2.5-flash",
                "tau_model":"gemini/gemini-2.5-flash",
            },
        ],
    },
    {
        "family":"Qwen",
        "label":"qwen",
        "routes":[
            {
                "provider":"groq",
                "base_url":GROQ_BASE,
                "api_key_name":"GROQ_API_KEY",
                "model":"qwen/qwen3.6-27b",
                "tau_model":"groq/qwen/qwen3.6-27b",
            },
            {
                "provider":"groq",
                "base_url":GROQ_BASE,
                "api_key_name":"GROQ_API_KEY",
                "model":"qwen/qwen3.8-27b",
                "tau_model":"groq/qwen/qwen3.8-27b",
            },
        ],
    },
]

def route_key(route):
    if route["api_key_name"]=="OPENAI_API_KEY":
        return OPENAI_API_KEY
    if route["api_key_name"]=="GEMINI_API_KEY":
        return GEMINI_API_KEY
    if route["api_key_name"]=="GROQ_API_KEY":
        return GROQ_API_KEY
    return ""

# Save only REDACTED provider metadata.
save_json(
    ENV/"provider_routes_redacted.json",
    [
        {
            "family":fam["family"],
            "label":fam["label"],
            "routes":[
                {
                    "provider":r["provider"],
                    "base_url":r["base_url"],
                    "model":r["model"],
                    "tau_model":r["tau_model"],
                }
                for r in fam["routes"]
            ],
        }
        for fam in FAMILY_ROUTES
    ],
)

display(pd.DataFrame([
    {
        "family":fam["family"],
        "routes":" → ".join(f"{r['provider']}:{r['model']}" for r in fam["routes"])
    }
    for fam in FAMILY_ROUTES
]))

In [ ]:
# ============================================================
# CELL 5 — CHAT + TOOL-CALL PREFLIGHT
# ============================================================
import urllib.request, urllib.error

TEST_TOOL=[{
    "type":"function",
    "function":{
        "name":"lookup_order",
        "description":"Lookup an order by ID",
        "parameters":{
            "type":"object",
            "properties":{"order_id":{"type":"string"}},
            "required":["order_id"],
        },
    },
}]

def chat_compatible(route,messages,tools=None):
    key=route_key(route)
    if not key:
        raise RuntimeError("NO_API_KEY_FOR_ROUTE")

    body={
        "model":route["model"],
        "messages":messages,
        "temperature":0,
    }

    if tools is not None:
        body["tools"]=tools
        # Required makes the preflight deterministic.
        body["tool_choice"]="required"

    req=urllib.request.Request(
        route["base_url"].rstrip("/")+"/chat/completions",
        data=json.dumps(body).encode(),
        headers={
            "Authorization":"Bearer "+key,
            "Content-Type":"application/json",
        },
        method="POST",
    )

    with urllib.request.urlopen(req,timeout=60) as response:
        return json.loads(response.read().decode())

preflight_rows=[]
SELECTED=[]

if SELFTEST:
    for fam in FAMILY_ROUTES:
        route=fam["routes"][0]
        preflight_rows.append({
            "family":fam["family"],
            "provider":route["provider"],
            "model":route["model"],
            "chat_ok":True,
            "tool_ok":True,
            "status":"SELFTEST_ONLY",
        })
        SELECTED.append({
            "family":fam["family"],
            "label":fam["label"],
            **route,
        })
else:
    for fam in FAMILY_ROUTES:
        chosen=None

        for route in fam["routes"]:
            row={
                "family":fam["family"],
                "provider":route["provider"],
                "model":route["model"],
            }

            try:
                result=chat_compatible(
                    route,
                    [{"role":"user","content":"Reply exactly OK."}],
                )
                row["chat_ok"]=bool(result.get("choices"))
            except Exception as e:
                row["chat_ok"]=False
                row["chat_error"]=repr(e)

            try:
                result=chat_compatible(
                    route,
                    [{"role":"user","content":"Call lookup_order for order A123."}],
                    tools=TEST_TOOL,
                )

                msg=(result.get("choices") or [{}])[0].get("message") or {}
                row["tool_ok"]=bool(msg.get("tool_calls"))

            except Exception as e:
                row["tool_ok"]=False
                row["tool_error"]=repr(e)

            row["status"]="OK" if row.get("chat_ok") and row.get("tool_ok") else "FAILED"
            preflight_rows.append(row)

            if row["status"]=="OK":
                chosen=route
                break

        if chosen:
            SELECTED.append({
                "family":fam["family"],
                "label":fam["label"],
                **chosen,
            })

preflight=pd.DataFrame(preflight_rows)
preflight.to_csv(
    RESULTS/"00_three_api_preflight.csv",
    index=False,
)

# No keys in this file.
save_json(
    ENV/"selected_routes_redacted.json",
    [
        {
            "family":r["family"],
            "label":r["label"],
            "provider":r["provider"],
            "base_url":r["base_url"],
            "model":r["model"],
            "tau_model":r["tau_model"],
        }
        for r in SELECTED
    ],
)

display(preflight)
print("\nSelected routes:")
for r in SELECTED:
    print(f"  {r['family']:<8} -> {r['provider']:<7} -> {r['model']}")

if not SELFTEST and len(SELECTED)<3:
    raise RuntimeError(
        f"Only {len(SELECTED)}/3 independent model families passed the API + tool-call preflight. "
        "Inspect 00_three_api_preflight.csv."
    )

In [ ]:
# ============================================================
# CELL 6 — IMPORT PREVIOUS MERGED RESULTS IF AVAILABLE
# ============================================================
candidate_paths=[]

if PREVIOUS_RESULTS_ZIP:
    candidate_paths.append(Path(PREVIOUS_RESULTS_ZIP))

candidate_paths += [
    Path("/content/NTX_RESULTS_MERGED_WITH_RECOVERY.zip"),
    Path.cwd()/"NTX_RESULTS_MERGED_WITH_RECOVERY.zip",
]

prev=next((p for p in candidate_paths if p.exists()),None)

if prev:
    dst=RAW/"previous_results"
    shutil.rmtree(dst,ignore_errors=True)
    dst.mkdir(parents=True,exist_ok=True)

    with zipfile.ZipFile(prev) as z:
        z.extractall(dst)

    previous={
        "status":"IMPORTED",
        "source":str(prev),
        "sha256":sha256_file(prev),
        "files":sum(1 for p in dst.rglob("*") if p.is_file()),
    }
else:
    previous={"status":"NOT_PROVIDED"}

save_json(
    RESULTS/"01_previous_results_import.json",
    previous,
)

print(previous)

# Stage A — BFCL-v4

The same benchmark is run once for each selected family, with that family's selected provider route.

In [ ]:
# ============================================================
# CELL 7 — BFCL ENVIRONMENT
# ============================================================
BFENV=WORK/"bfcl_env"

if SELFTEST:
    BFPY=Path(sys.executable)
else:
    ensure_uv()
    run(["uv","python","install","3.11"])

    if not BFENV.exists():
        p=run([
            "uv","venv",BFENV,
            "--python","3.11",
        ])
        save_log("bfcl_venv.log",p)

        if p.returncode:
            raise RuntimeError("BFCL virtual environment creation failed.")

    BFPY=BFENV/"bin"/"python"

    p=run([
        "uv","pip","install",
        "--python",BFPY,
        "-U",
        "evalscope[bfcl]",
    ])
    save_log("bfcl_install.log",p)

    if p.returncode:
        raise RuntimeError(
            "BFCL install failed. Inspect bfcl_install.log."
        )

    verify=run([
        BFPY,
        "-c",
        "from evalscope import run_task; "
        "from evalscope.config import TaskConfig; "
        "print('BFCL_READY')",
    ])
    save_log("bfcl_verify.log",verify)

    if verify.returncode or "BFCL_READY" not in verify.stdout:
        raise RuntimeError("BFCL environment verification failed.")

    freeze=run([BFPY,"-m","pip","freeze"])
    (ENV/"bfcl_pip_freeze.txt").write_text(
        freeze.stdout or ""
    )

print("BFCL environment ready")

In [ ]:
# ============================================================
# CELL 8 — BFCL EXECUTION + STRICT PARSER
# ============================================================
def parse_bfcl(root,label,family):
    root=Path(root)
    rows=[]
    evaluated=0

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        rel=str(p.relative_to(root))
        low=rel.lower()

        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)

                if any(
                    x in low
                    for x in ["report","review","prediction"]
                ):
                    evaluated=max(
                        evaluated,
                        len(df),
                    )

                for col in df.columns:
                    if any(
                        x in str(col).lower()
                        for x in ["accuracy","score"]
                    ):
                        vals=pd.to_numeric(
                            df[col],
                            errors="coerce",
                        ).dropna()

                        for value in vals:
                            rows.append({
                                "benchmark":"BFCL-v4",
                                "model":label,
                                "family":family,
                                "slice":rel,
                                "metric":str(col),
                                "score":float(value),
                                "source_file":rel,
                            })

            elif p.suffix.lower() in {".json",".jsonl"}:
                texts=(
                    p.read_text(
                        errors="ignore"
                    ).splitlines()
                    if p.suffix.lower()==".jsonl"
                    else [
                        p.read_text(
                            errors="ignore"
                        )
                    ]
                )

                for text in texts:
                    try:
                        obj=json.loads(text)
                    except Exception:
                        continue

                    stack=[("",obj)]

                    while stack:
                        path,o=stack.pop()

                        if isinstance(o,dict):
                            for count_key in [
                                "total_count",
                                "evaluated_count",
                                "num_samples",
                                "n_samples",
                            ]:
                                if (
                                    isinstance(
                                        o.get(count_key),
                                        (int,float),
                                    )
                                    and o[count_key]>0
                                ):
                                    evaluated=max(
                                        evaluated,
                                        int(o[count_key]),
                                    )

                            for key,value in o.items():
                                child=(
                                    f"{path}.{key}"
                                    if path
                                    else key
                                )

                                if isinstance(
                                    value,
                                    (dict,list),
                                ):
                                    stack.append(
                                        (child,value)
                                    )

                                elif (
                                    isinstance(
                                        value,
                                        (int,float),
                                    )
                                    and any(
                                        x in str(key).lower()
                                        for x in [
                                            "accuracy",
                                            "score",
                                        ]
                                    )
                                ):
                                    rows.append({
                                        "benchmark":"BFCL-v4",
                                        "model":label,
                                        "family":family,
                                        "slice":rel,
                                        "metric":child,
                                        "score":float(value),
                                        "source_file":rel,
                                    })

                        elif isinstance(o,list):
                            if (
                                any(
                                    x in low
                                    for x in [
                                        "review",
                                        "prediction",
                                    ]
                                )
                                and o
                            ):
                                evaluated=max(
                                    evaluated,
                                    len(o),
                                )

                            for i,value in enumerate(o):
                                stack.append(
                                    (
                                        f"{path}[{i}]",
                                        value,
                                    )
                                )
        except Exception:
            continue

    df=(
        pd.DataFrame(rows).drop_duplicates()
        if rows
        else pd.DataFrame()
    )

    return df,evaluated

bfcl_parts=[]
bfcl_status_rows=[]

if SELFTEST:
    for spec in SELECTED:
        root=(
            RAW/"bfcl"/
            spec["label"]/
            "selftest"
        )
        root.mkdir(
            parents=True,
            exist_ok=True,
        )

        (
            root/"report.json"
        ).write_text(
            json.dumps({
                "accuracy":0.80,
                "total_count":5,
            })
        )

        parsed,n=parse_bfcl(
            root,
            spec["label"],
            spec["family"],
        )

        parsed["evidence_state"]="SELFTEST_ONLY"
        bfcl_parts.append(parsed)

        bfcl_status_rows.append({
            "family":spec["family"],
            "provider":spec["provider"],
            "model_id":spec["model"],
            "status":"SELFTEST_ONLY",
            "evaluated":n,
        })

else:
    for spec in SELECTED:
        key=(
            f"bfcl::{spec['family']}::{MODE}"
        )

        root=(
            RAW/"bfcl"/
            spec["label"]/
            MODE.lower()
        )
        root.mkdir(
            parents=True,
            exist_ok=True,
        )

        api_key=route_key(spec)

        script=root/"run_bfcl.py"

        # Key is used by the subprocess at runtime but the generated
        # script is deleted immediately after execution and is never
        # copied to the final archives.
        script.write_text(
f'''from evalscope import run_task
from evalscope.config import TaskConfig

cfg=TaskConfig(
    model={spec["model"]!r},
    api_url={spec["base_url"]!r},
    api_key={api_key!r},
    eval_type="openai_api",
    datasets=["bfcl_v4"],
    work_dir={str(root)!r},
    limit={CFG["bfcl_limit"]!r},
    seed={SEED},
    generation_config={{
        "temperature":0.0,
        "max_tokens":1024,
        "retries":1,
        "retry_interval":2,
        "timeout":120,
    }},
    dataset_args={{
        "bfcl_v4": {{
            "extra_params": {{
                "is_fc_model": True
            }}
        }}
    }},
)

run_task(task_cfg=cfg)
''')

        p=run(
            [BFPY,script],
            cwd=root,
            timeout=None,
        )

        # Do not include command because script path is enough and the
        # script temporarily contained the secret.
        save_log(
            f"bfcl_{spec['label']}.log",
            p,
        )

        try:
            script.unlink()
        except Exception:
            pass

        parsed,n=parse_bfcl(
            root,
            spec["label"],
            spec["family"],
        )

        status=(
            "SUPPORTED"
            if n>0 and len(parsed)>0
            else "INFRA_FAILURE"
        )

        if len(parsed):
            parsed["provider"]=spec["provider"]
            parsed["model_id"]=spec["model"]
            bfcl_parts.append(parsed)

        bfcl_status_rows.append({
            "family":spec["family"],
            "provider":spec["provider"],
            "model_id":spec["model"],
            "status":status,
            "evaluated":n,
            "returncode":p.returncode,
        })

        ck_set(
            key,
            status,
            provider=spec["provider"],
            model_id=spec["model"],
            evaluated=n,
        )

bfcl=(
    pd.concat(
        bfcl_parts,
        ignore_index=True,
    )
    if bfcl_parts
    else pd.DataFrame()
)

bfcl_status=pd.DataFrame(
    bfcl_status_rows
)

bfcl.to_csv(
    RESULTS/"10_bfcl_metrics.csv",
    index=False,
)

bfcl_status.to_csv(
    RESULTS/"10_bfcl_status.csv",
    index=False,
)

display(bfcl_status)

# Stage B — AgentDojo

AgentDojo is run once per selected provider/model route using its `openai-compatible` adapter.

In [ ]:
# ============================================================
# CELL 9 — AGENTDOJO ENVIRONMENT
# ============================================================
DOJO=WORK/"agentdojo"

if SELFTEST:
    DOJO_COMMIT="SELFTEST"
else:
    DOJO_COMMIT,git_status=clone(
        "https://github.com/ethz-spylab/agentdojo.git",
        DOJO,
    )

    (
        ENV/"agentdojo_git_status.txt"
    ).write_text(
        git_status
    )

    ensure_uv()

    p=run(
        ["uv","sync"],
        cwd=DOJO,
    )
    save_log(
        "agentdojo_sync.log",
        p,
    )

    if p.returncode:
        raise RuntimeError(
            "AgentDojo environment setup failed."
        )

    help_proc=run(
        [
            "uv","run","python",
            "-m",
            "agentdojo.scripts.benchmark",
            "--help",
        ],
        cwd=DOJO,
    )

    save_log(
        "agentdojo_help.log",
        help_proc,
    )

    text=(
        (help_proc.stdout or "")
        +
        (help_proc.stderr or "")
    )

    required=[
        "openai-compatible",
        "--model-id",
        "--force-rerun",
    ]

    if not all(
        item in text
        for item in required
    ):
        raise RuntimeError(
            "Installed AgentDojo does not expose "
            "the required current openai-compatible interface."
        )

    freeze=run(
        ["uv","pip","freeze"],
        cwd=DOJO,
    )

    (
        ENV/"agentdojo_pip_freeze.txt"
    ).write_text(
        freeze.stdout or ""
    )

save_json(
    ENV/"agentdojo_version.json",
    {"commit":DOJO_COMMIT},
)

print("AgentDojo commit:",DOJO_COMMIT)

In [ ]:
# ============================================================
# CELL 10 — AGENTDOJO RUN + PARSER
# ============================================================
def parse_dojo(root,spec,suite):
    root=Path(root)
    rows=[]

    for p in root.rglob("*.json"):
        try:
            obj=json.loads(
                p.read_text()
            )
        except Exception:
            continue

        if not isinstance(obj,dict):
            continue

        utility=obj.get("utility")
        security=obj.get("security")

        if (
            not isinstance(utility,bool)
            and
            not isinstance(security,bool)
        ):
            continue

        rows.append({
            "benchmark":"AgentDojo",
            "model":spec["label"],
            "family":spec["family"],
            "provider":spec["provider"],
            "model_id":spec["model"],
            "suite":suite,
            "user_task_id":obj.get(
                "user_task_id"
            ),
            "injection_task_id":obj.get(
                "injection_task_id"
            ),
            "utility":(
                np.nan
                if not isinstance(
                    utility,
                    bool,
                )
                else int(utility)
            ),
            "security":(
                np.nan
                if not isinstance(
                    security,
                    bool,
                )
                else int(security)
            ),
            "error":obj.get("error"),
            "source_file":str(
                p.relative_to(root)
            ),
        })

    return pd.DataFrame(rows)

dojo_parts=[]
dojo_status_rows=[]

if SELFTEST:
    for spec in SELECTED:
        suite="banking"

        root=(
            RAW/"agentdojo"/
            spec["label"]/
            suite/
            "selftest"
        )
        root.mkdir(
            parents=True,
            exist_ok=True,
        )

        (
            root/"case.json"
        ).write_text(
            json.dumps({
                "suite_name":suite,
                "user_task_id":"user_task_0",
                "injection_task_id":"injection_task_0",
                "utility":True,
                "security":True,
                "error":None,
            })
        )

        data=parse_dojo(
            root,
            spec,
            suite,
        )

        data["evidence_state"]="SELFTEST_ONLY"
        dojo_parts.append(data)

        dojo_status_rows.append({
            "family":spec["family"],
            "provider":spec["provider"],
            "suite":suite,
            "status":"SELFTEST_ONLY",
            "n_scored":len(data),
        })

else:
    for spec in SELECTED:
        provider_env=os.environ.copy()

        provider_env[
            "OPENAI_COMPATIBLE_BASE_URL"
        ]=spec["base_url"]

        provider_env[
            "OPENAI_COMPATIBLE_API_KEY"
        ]=route_key(spec)

        for suite in CFG["dojo_suites"]:
            root=(
                RAW/"agentdojo"/
                spec["label"]/
                suite/
                MODE.lower()
            )
            root.mkdir(
                parents=True,
                exist_ok=True,
            )

            cmd=[
                "uv","run","python",
                "-m",
                "agentdojo.scripts.benchmark",
                "--model",
                "openai-compatible",
                "--model-id",
                spec["model"],
                "-s",
                suite,
                "--attack",
                "important_instructions",
                "--logdir",
                str(root),
                "--force-rerun",
                "--max-workers",
                "1",
            ]

            if (
                CFG["dojo_user_tasks"]
                is not None
            ):
                for task_id in CFG[
                    "dojo_user_tasks"
                ]:
                    cmd += [
                        "-ut",
                        task_id,
                    ]

            if (
                CFG[
                    "dojo_injection_tasks"
                ]
                is not None
            ):
                for task_id in CFG[
                    "dojo_injection_tasks"
                ]:
                    cmd += [
                        "-it",
                        task_id,
                    ]

            p=run(
                cmd,
                cwd=DOJO,
                env=provider_env,
                timeout=None,
            )

            save_log(
                f"dojo_{spec['label']}_{suite}.log",
                p,
                cmd,
            )

            data=parse_dojo(
                root,
                spec,
                suite,
            )

            n_valid=(
                int(
                    (
                        data.utility.notna()
                        |
                        data.security.notna()
                    ).sum()
                )
                if len(data)
                else 0
            )

            errors=(
                int(
                    data.error.notna().sum()
                )
                if len(data)
                else 0
            )

            status=(
                "SUPPORTED"
                if n_valid>0 and errors==0
                else (
                    "PARTIAL"
                    if n_valid>0
                    else "INFRA_FAILURE"
                )
            )

            if len(data):
                dojo_parts.append(data)

            dojo_status_rows.append({
                "family":spec["family"],
                "provider":spec["provider"],
                "model_id":spec["model"],
                "suite":suite,
                "status":status,
                "n_scored":n_valid,
                "errors":errors,
                "returncode":p.returncode,
            })

            ck_set(
                f"dojo::{spec['family']}::{suite}::{MODE}",
                status,
                provider=spec["provider"],
                model_id=spec["model"],
                n_scored=n_valid,
            )

dojo=(
    pd.concat(
        dojo_parts,
        ignore_index=True,
    )
    if dojo_parts
    else pd.DataFrame()
)

dojo_status=pd.DataFrame(
    dojo_status_rows
)

dojo.to_csv(
    RESULTS/"11_agentdojo_cases.csv",
    index=False,
)

dojo_status.to_csv(
    RESULTS/"11_agentdojo_status.csv",
    index=False,
)

display(dojo_status)

# Stage C — τ³ / tau2-bench

LiteLLM uses provider-specific model IDs:

- OpenAI: `openai/...`
- Gemini: `gemini/...`
- Groq: `groq/...`

A fixed Gemini Flash-Lite user simulator is used when possible, so the simulated user is consistent across agent families.

In [ ]:
# ============================================================
# CELL 11 — TAU ENVIRONMENT
# ============================================================
TAU=WORK/"tau2-bench"

if SELFTEST:
    TAU_COMMIT="SELFTEST"
else:
    TAU_COMMIT,tau_git_status=clone(
        "https://github.com/sierra-research/tau2-bench.git",
        TAU,
    )

    (
        ENV/"tau_git_status.txt"
    ).write_text(
        tau_git_status
    )

    ensure_uv()

    p=run(
        ["uv","sync"],
        cwd=TAU,
    )
    save_log(
        "tau_sync.log",
        p,
    )

    if p.returncode:
        raise RuntimeError(
            "tau2 setup failed."
        )

    TAUPY=TAU/".venv"/"bin"/"python"

    p=run(
        [
            "uv","pip","install",
            "--python",TAUPY,
            "websockets",
            "soundfile",
        ],
        cwd=TAU,
    )
    save_log(
        "tau_dependencies.log",
        p,
    )

    verify=run(
        [
            TAUPY,
            "-c",
            "import websockets,soundfile; "
            "print('TAU_READY')",
        ],
        cwd=TAU,
    )

    save_log(
        "tau_verify.log",
        verify,
    )

    if (
        verify.returncode
        or
        "TAU_READY" not in verify.stdout
    ):
        raise RuntimeError(
            "tau2 dependency verification failed."
        )

    freeze=run(
        ["uv","pip","freeze"],
        cwd=TAU,
    )

    (
        ENV/"tau_pip_freeze.txt"
    ).write_text(
        freeze.stdout or ""
    )

save_json(
    ENV/"tau_version.json",
    {"commit":TAU_COMMIT},
)

print("tau2 commit:",TAU_COMMIT)

In [ ]:
# ============================================================
# CELL 12 — TAU RUN + STRICT NATIVE PARSER
# ============================================================
def parse_tau(path):
    try:
        root=json.loads(
            Path(path).read_text()
        )
    except Exception:
        return pd.DataFrame()

    if isinstance(root,list):
        simulations=root
    elif isinstance(root,dict):
        simulations=next(
            (
                root[k]
                for k in [
                    "simulations",
                    "results",
                    "trajectories",
                ]
                if isinstance(
                    root.get(k),
                    list,
                )
            ),
            [],
        )
    else:
        simulations=[]

    rows=[]

    for i,item in enumerate(simulations):
        if not isinstance(item,dict):
            continue

        reward=None

        if (
            isinstance(
                item.get("reward_info"),
                dict,
            )
            and isinstance(
                item["reward_info"].get(
                    "reward"
                ),
                (int,float,bool),
            )
        ):
            reward=float(
                item["reward_info"]["reward"]
            )

        elif isinstance(
            item.get("reward"),
            (int,float,bool),
        ):
            reward=float(
                item["reward"]
            )

        error=item.get("error")

        if (
            error is None
            and isinstance(
                item.get("info"),
                dict,
            )
        ):
            error=item["info"].get(
                "error"
            )

        rows.append({
            "trajectory_index":i,
            "task_id":item.get("task_id"),
            "trial":item.get("trial"),
            "reward":reward,
            "error":error,
        })

    return pd.DataFrame(rows)

# Prefer Gemini Flash-Lite as a common user simulator.
GEMINI_SELECTED=next(
    (
        x
        for x in SELECTED
        if x["family"]=="Gemini"
    ),
    None,
)

FIXED_USER_MODEL=(
    GEMINI_SELECTED["tau_model"]
    if GEMINI_SELECTED
    else SELECTED[0]["tau_model"]
)

tau_parts=[]
tau_status_rows=[]

if SELFTEST:
    for spec in SELECTED:
        for domain in CFG["tau_domains"]:
            root=(
                RAW/"tau"/
                spec["label"]/
                domain/
                "selftest"
            )
            root.mkdir(
                parents=True,
                exist_ok=True,
            )

            (
                root/"results.json"
            ).write_text(
                json.dumps({
                    "simulations":[
                        {
                            "task_id":"0",
                            "trial":0,
                            "reward_info":{
                                "reward":1.0
                            },
                            "error":None,
                        },
                        {
                            "task_id":"1",
                            "trial":0,
                            "reward_info":{
                                "reward":0.0
                            },
                            "error":None,
                        },
                    ]
                })
            )

            data=parse_tau(
                root/"results.json"
            )

            data["benchmark"]="tau3"
            data["model"]=spec["label"]
            data["family"]=spec["family"]
            data["domain"]=domain
            data["evidence_state"]="SELFTEST_ONLY"

            tau_parts.append(data)

            tau_status_rows.append({
                "family":spec["family"],
                "provider":spec["provider"],
                "domain":domain,
                "status":"SELFTEST_ONLY",
                "n_evaluated":int(
                    data.reward.notna().sum()
                ),
            })

else:
    common_env=os.environ.copy()

    if OPENAI_API_KEY:
        common_env[
            "OPENAI_API_KEY"
        ]=OPENAI_API_KEY

    common_env[
        "GEMINI_API_KEY"
    ]=GEMINI_API_KEY

    common_env[
        "GROQ_API_KEY"
    ]=GROQ_API_KEY

    for spec in SELECTED:
        for domain in CFG["tau_domains"]:
            run_name=(
                f"ntx_3api_"
                f"{spec['label']}_"
                f"{domain}_"
                f"{MODE.lower()}"
            )

            live_dir=(
                TAU/
                "data"/
                "simulations"/
                run_name
            )

            archive_dir=(
                RAW/"tau"/
                spec["label"]/
                domain/
                MODE.lower()
            )

            cmd=[
                "uv","run",
                "tau2","run",

                "--domain",
                domain,

                "--agent-llm",
                spec["tau_model"],

                "--user-llm",
                FIXED_USER_MODEL,

                "--agent-llm-args",
                json.dumps({
                    "temperature":0.0,
                    "max_tokens":
                        CFG["agent_max_tokens"],
                }),

                "--user-llm-args",
                json.dumps({
                    "temperature":0.0,
                    "max_tokens":
                        CFG["user_max_tokens"],
                }),

                "--num-trials",
                "1",

                "--task-split-name",
                "base",

                "--max-steps",
                str(CFG["tau_max_steps"]),

                "--max-errors",
                "3",

                "--max-concurrency",
                "1",

                "--seed",
                str(SEED),

                "--save-to",
                run_name,

                "--verbose-logs",

                "--llm-log-mode",
                "all",
            ]

            if (
                CFG["tau_num_tasks"]
                is not None
            ):
                cmd += [
                    "--num-tasks",
                    str(
                        CFG[
                            "tau_num_tasks"
                        ]
                    ),
                ]

            p=run(
                cmd,
                cwd=TAU,
                env=common_env,
                timeout=None,
            )

            save_log(
                f"tau_{spec['label']}_{domain}.log",
                p,
                cmd,
            )

            if live_dir.exists():
                copytree(
                    live_dir,
                    archive_dir,
                )

            results_file=(
                archive_dir/
                "results.json"
            )

            data=(
                parse_tau(results_file)
                if results_file.exists()
                else pd.DataFrame()
            )

            n_evaluated=(
                int(
                    data.reward.notna().sum()
                )
                if len(data)
                else 0
            )

            infrastructure_errors=(
                int(
                    data.error.notna().sum()
                )
                if len(data)
                else 0
            )

            status=(
                "SUPPORTED"
                if (
                    n_evaluated>0
                    and
                    infrastructure_errors==0
                )
                else (
                    "PARTIAL"
                    if n_evaluated>0
                    else "INFRA_FAILURE"
                )
            )

            if len(data):
                data["benchmark"]="tau3"
                data["model"]=spec["label"]
                data["family"]=spec["family"]
                data["provider"]=spec["provider"]
                data["model_id"]=spec["model"]
                data["domain"]=domain

                tau_parts.append(data)

            tau_status_rows.append({
                "family":spec["family"],
                "provider":spec["provider"],
                "model_id":spec["model"],
                "domain":domain,
                "status":status,
                "n_evaluated":n_evaluated,
                "infra_errors":
                    infrastructure_errors,
                "returncode":p.returncode,
            })

            ck_set(
                f"tau::{spec['family']}::{domain}::{MODE}",
                status,
                provider=spec["provider"],
                model_id=spec["model"],
                n_evaluated=n_evaluated,
            )

tau=(
    pd.concat(
        tau_parts,
        ignore_index=True,
    )
    if tau_parts
    else pd.DataFrame()
)

tau_status=pd.DataFrame(
    tau_status_rows
)

tau.to_csv(
    RESULTS/"14_tau_cases.csv",
    index=False,
)

tau_status.to_csv(
    RESULTS/"14_tau_status.csv",
    index=False,
)

display(tau_status)

In [ ]:
# ============================================================
# CELL 13 — UNIFIED EXTERNAL EVIDENCE
# ============================================================
rows=[]

if len(bfcl):
    for _,r in bfcl.iterrows():
        rows.append({
            "benchmark":"BFCL-v4",
            "family":r.get("family"),
            "model":r.get("model"),
            "provider":r.get(
                "provider"
            ),
            "slice":r.get("slice"),
            "metric":r.get("metric"),
            "score":r.get("score"),
            "evidence_state":(
                "SELFTEST_ONLY"
                if SELFTEST
                else "SUPPORTED"
            ),
        })

if len(dojo):
    for _,r in dojo.iterrows():
        if pd.notna(
            r.get("utility")
        ):
            rows.append({
                "benchmark":"AgentDojo",
                "family":r["family"],
                "model":r["model"],
                "provider":r["provider"],
                "slice":r["suite"],
                "metric":"utility",
                "score":float(
                    r["utility"]
                ),
                "evidence_state":(
                    "SELFTEST_ONLY"
                    if SELFTEST
                    else "SUPPORTED"
                ),
            })

        if pd.notna(
            r.get("security")
        ):
            rows.append({
                "benchmark":"AgentDojo",
                "family":r["family"],
                "model":r["model"],
                "provider":r["provider"],
                "slice":r["suite"],
                "metric":"security",
                "score":float(
                    r["security"]
                ),
                "evidence_state":(
                    "SELFTEST_ONLY"
                    if SELFTEST
                    else "SUPPORTED"
                ),
            })

if len(tau):
    for _,r in tau.iterrows():
        if pd.notna(
            r.get("reward")
        ):
            rows.append({
                "benchmark":"tau3",
                "family":r["family"],
                "model":r["model"],
                "provider":r.get(
                    "provider"
                ),
                "slice":r["domain"],
                "metric":"reward",
                "score":float(
                    r["reward"]
                ),
                "evidence_state":(
                    "SELFTEST_ONLY"
                    if SELFTEST
                    else "SUPPORTED"
                ),
            })

evidence=pd.DataFrame(rows)

evidence.to_csv(
    RESULTS/
    "20_unified_external_evidence.csv",
    index=False,
)

if len(evidence):
    summary=(
        evidence.groupby(
            [
                "benchmark",
                "family",
                "model",
                "provider",
                "slice",
                "metric",
                "evidence_state",
            ],
            dropna=False,
        )
        .agg(
            n_cases=("score","count"),
            mean_score=("score","mean"),
            min_score=("score","min"),
            max_score=("score","max"),
        )
        .reset_index()
    )
else:
    summary=pd.DataFrame()

summary.to_csv(
    RESULTS/
    "20_external_summary.csv",
    index=False,
)

display(summary)

In [ ]:
# ============================================================
# CELL 14 — BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================
def bootstrap_ci(values,B):
    x=np.asarray(
        pd.Series(values).dropna(),
        dtype=float,
    )

    if len(x)==0:
        return np.nan,np.nan,np.nan

    if len(x)==1:
        return float(x[0]),np.nan,np.nan

    rng=np.random.default_rng(SEED)
    means=np.empty(B)

    for i in range(B):
        means[i]=rng.choice(
            x,
            size=len(x),
            replace=True,
        ).mean()

    return (
        float(x.mean()),
        float(
            np.quantile(
                means,
                0.025,
            )
        ),
        float(
            np.quantile(
                means,
                0.975,
            )
        ),
    )

ci_rows=[]

if len(evidence):
    for (
        benchmark,
        family,
        slice_name,
        metric,
    ),group in evidence.groupby(
        [
            "benchmark",
            "family",
            "slice",
            "metric",
        ]
    ):
        mean,lo,hi=bootstrap_ci(
            group.score,
            CFG["bootstrap"],
        )

        ci_rows.append({
            "benchmark":benchmark,
            "family":family,
            "slice":slice_name,
            "metric":metric,
            "n":len(group),
            "mean":mean,
            "ci95_low":lo,
            "ci95_high":hi,
        })

ci=pd.DataFrame(ci_rows)

ci.to_csv(
    RESULTS/"21_bootstrap_ci.csv",
    index=False,
)

display(ci)

In [ ]:
# ============================================================
# CELL 15 — STRICT PAPER-CLOSURE CLAIM GATE
# ============================================================
REQUIRED_FAMILIES={
    "OpenAI",
    "Gemini",
    "Qwen",
}

bf_ok=(
    not SELFTEST
    and len(bfcl_status)>0
    and REQUIRED_FAMILIES.issubset(
        set(
            bfcl_status.loc[
                bfcl_status.status=="SUPPORTED",
                "family",
            ].astype(str)
        )
    )
)

dojo_ok=False

if (
    not SELFTEST
    and len(dojo_status)
):
    dojo_ok=True

    for family in REQUIRED_FAMILIES:
        have=set(
            dojo_status.loc[
                (
                    dojo_status.family==family
                )
                &
                (
                    dojo_status.status==
                    "SUPPORTED"
                ),
                "suite",
            ].astype(str)
        )

        if not set(
            CFG["dojo_suites"]
        ).issubset(have):
            dojo_ok=False
            break

tau_ok=False

if (
    not SELFTEST
    and len(tau_status)
):
    tau_ok=True

    for family in REQUIRED_FAMILIES:
        have=set(
            tau_status.loc[
                (
                    tau_status.family==family
                )
                &
                (
                    tau_status.status==
                    "SUPPORTED"
                ),
                "domain",
            ].astype(str)
        )

        if not set(
            CFG["tau_domains"]
        ).issubset(have):
            tau_ok=False
            break

claims=pd.DataFrame([
    {
        "claim":"BFCL-v4 closure",
        "status":(
            "SELFTEST_ONLY"
            if SELFTEST
            else (
                "SUPPORTED"
                if bf_ok
                else "MISSING"
            )
        ),
    },
    {
        "claim":"AgentDojo closure",
        "status":(
            "SELFTEST_ONLY"
            if SELFTEST
            else (
                "SUPPORTED"
                if dojo_ok
                else "MISSING"
            )
        ),
    },
    {
        "claim":"tau3 closure",
        "status":(
            "SELFTEST_ONLY"
            if SELFTEST
            else (
                "SUPPORTED"
                if tau_ok
                else (
                    "PARTIAL"
                    if len(tau)
                    else "MISSING"
                )
            )
        ),
    },
    {
        "claim":"three independent model families",
        "status":(
            "SELFTEST_ONLY"
            if SELFTEST
            else (
                "SUPPORTED"
                if {
                    x["family"]
                    for x in SELECTED
                } == REQUIRED_FAMILIES
                else "MISSING"
            )
        ),
    },
    {
        "claim":"External-validation closure gate",
        "status":(
            "SELFTEST_ONLY"
            if SELFTEST
            else (
                "SUPPORTED"
                if (
                    bf_ok
                    and dojo_ok
                    and tau_ok
                )
                else "INCOMPLETE"
            )
        ),
    },
])

claims.to_csv(
    RESULTS/
    "22_paper_closure_claims.csv",
    index=False,
)

display(claims)

In [ ]:
# ============================================================
# CELL 16 — PAPER-INTEGRATION TABLES + TEXT
# ============================================================
if len(summary):
    summary.to_latex(
        PAPER/
        "external_benchmark_summary.tex",
        index=False,
        float_format="%.4f",
    )

if len(ci):
    ci.to_latex(
        PAPER/
        "external_bootstrap_ci.tex",
        index=False,
        float_format="%.4f",
    )

claims.to_latex(
    PAPER/
    "external_claim_gate.tex",
    index=False,
)

compact=[]

if len(evidence):
    for (
        benchmark,
        family,
        metric,
    ),group in evidence.groupby(
        [
            "benchmark",
            "family",
            "metric",
        ]
    ):
        compact.append({
            "benchmark":benchmark,
            "family":family,
            "metric":metric,
            "n":len(group),
            "mean":float(
                group.score.mean()
            ),
        })

compact_df=pd.DataFrame(compact)

compact_df.to_csv(
    PAPER/
    "external_compact_table.csv",
    index=False,
)

if len(compact_df):
    compact_df.to_latex(
        PAPER/
        "external_compact_table.tex",
        index=False,
        float_format="%.4f",
    )

gate=claims.loc[
    claims.claim==
    "External-validation closure gate",
    "status",
].iloc[0]

if SELFTEST:
    section=(
        "\\paragraph{External validation.}\n"
        "NON-PAPER SELFTEST output. "
        "Do not include this paragraph "
        "in a manuscript.\n"
    )

elif gate=="SUPPORTED":
    section=(
        "\\paragraph{External validation.}\n"
        "We evaluated NiyamTrace-X on three independent "
        "external benchmark families: BFCL-v4 for function "
        "calling, AgentDojo for prompt-injection robustness, "
        "and $\\tau^3$ for stateful customer-service tool use. "
        "The matrix covers three independent model families "
        "(OpenAI, Gemini, and Qwen) using provider-native or "
        "OpenAI-compatible interfaces. The reported rows use "
        "benchmark-native metrics and are not pooled into a "
        "single heterogeneous score.\n"
    )

else:
    available=(
        sorted(
            evidence.benchmark.unique()
        )
        if len(evidence)
        else []
    )

    section=(
        "\\paragraph{External validation.}\n"
        "The external-validation matrix remains incomplete. "
        "Native evidence is currently available for "
        + (
            ", ".join(available)
            if available
            else "none of the required external benchmarks"
        )
        + ". We therefore retain the frozen internal "
        "evaluation as the primary effectiveness claim and "
        "treat available external results as bounded transfer "
        "evidence only.\n"
    )

(
    PAPER/
    "external_validation_section.tex"
).write_text(section)

(
    PAPER/
    "README.md"
).write_text(
    "# NiyamTrace-X external paper integration\n\n"
    f"Closure gate: **{gate}**\n\n"
    "Generated directly from benchmark-native outputs.\n"
)

print((PAPER/"README.md").read_text())

In [ ]:
# ============================================================
# CELL 17 — FIGURES
# ============================================================
if len(summary):
    for (
        benchmark,
        metric,
    ),group in summary.groupby(
        ["benchmark","metric"]
    ):

        chart_rows=[]

        for family,family_group in group.groupby(
            "family"
        ):
            weights=np.maximum(
                family_group[
                    "n_cases"
                ].to_numpy(
                    dtype=float
                ),
                1,
            )

            score=float(
                np.average(
                    family_group[
                        "mean_score"
                    ].to_numpy(
                        dtype=float
                    ),
                    weights=weights,
                )
            )

            chart_rows.append({
                "family":family,
                "score":score,
            })

        chart=pd.DataFrame(
            chart_rows
        ).sort_values("score")

        if not len(chart):
            continue

        fig,ax=plt.subplots(
            figsize=(
                8,
                max(
                    3,
                    0.55*len(chart)+1,
                ),
            )
        )

        ax.barh(
            chart["family"],
            chart["score"],
        )

        ax.set_title(
            f"{benchmark}: {metric}"
        )

        ax.set_xlabel(metric)

        fig.tight_layout()

        filename=re.sub(
            r"[^A-Za-z0-9]+",
            "_",
            f"{benchmark}_{metric}",
        ).strip("_")

        fig.savefig(
            PAPER/f"{filename}.png",
            dpi=240,
            bbox_inches="tight",
        )

        plt.show()

In [ ]:
# ============================================================
# CELL 18 — PROVENANCE / SECRET-SCAN / MANIFEST
# ============================================================
# Explicitly remove any accidental secret-bearing generated scripts.
for p in BASE.rglob("run_bfcl.py"):
    try:
        p.unlink()
    except Exception:
        pass

# Search text artifacts for exact API key values.
secret_values=[
    x for x in [
        GEMINI_API_KEY,
        OPENAI_API_KEY,
        GROQ_API_KEY,
    ]
    if x and not x.startswith("SELFTEST")
]

secret_hits=[]

for root in [
    RESULTS,
    RAW,
    LOGS,
    ENV,
    PAPER,
]:
    for p in root.rglob("*"):
        if not p.is_file():
            continue

        if p.suffix.lower() not in {
            ".txt",
            ".log",
            ".json",
            ".jsonl",
            ".csv",
            ".md",
            ".tex",
            ".py",
        }:
            continue

        try:
            text=p.read_text(
                errors="ignore"
            )
        except Exception:
            continue

        for secret in secret_values:
            if secret in text:
                secret_hits.append(
                    str(p)
                )

if secret_hits:
    raise RuntimeError(
        "SECURITY STOP: API key detected "
        "inside generated artifacts: "
        + "; ".join(
            sorted(
                set(secret_hits)
            )
        )
    )

manifest={
    "experiment":
        "NTX-THREE-API-PAPER-CLOSURE",
    "created_at":now(),
    "mode":MODE,
    "selftest":SELFTEST,
    "selected_routes":[
        {
            "family":x["family"],
            "provider":x["provider"],
            "model":x["model"],
            "tau_model":x["tau_model"],
        }
        for x in SELECTED
    ],
    "claims":
        claims.to_dict("records"),
    "previous_results":
        previous,
    "config":
        CFG,
    "secret_scan":
        "PASS",
}

hashes={}

for root_name,root in [
    ("results",RESULTS),
    ("raw",RAW),
    ("logs",LOGS),
    ("environment",ENV),
    ("paper",PAPER),
]:
    for p in sorted(
        root.rglob("*")
    ):
        if (
            p.is_file()
            and p.name not in {
                "FINAL_MANIFEST.json",
                "SHA256SUMS.txt",
            }
        ):
            hashes[
                f"{root_name}/"
                f"{p.relative_to(root)}"
            ]=sha256_file(p)

manifest[
    "artifact_sha256"
]=hashes

save_json(
    RESULTS/"FINAL_MANIFEST.json",
    manifest,
)

(
    RESULTS/
    "SHA256SUMS.txt"
).write_text(
    "\n".join(
        f"{hash_value}  {name}"
        for name,hash_value
        in sorted(
            hashes.items()
        )
    )
    + "\n"
)

print("Secret scan: PASS")
print("Manifest artifacts:",len(hashes))

In [ ]:
# ============================================================
# CELL 19 — FINAL FOUR ZIP ARCHIVES
# ============================================================
raw_stage=BASE/"_raw_export"
shutil.rmtree(
    raw_stage,
    ignore_errors=True,
)
raw_stage.mkdir()

copytree(
    RAW,
    raw_stage/"benchmark_raw",
)
copytree(
    LOGS,
    raw_stage/"logs",
)
copytree(
    ENV,
    raw_stage/"environment",
)

RAW_ZIP=ARCH/"NTX_3API_RAW_DATA.zip"
RESULTS_ZIP=ARCH/"NTX_3API_PROCESSED_RESULTS.zip"
PAPER_ZIP=ARCH/"NTX_3API_PAPER_INTEGRATION.zip"
MASTER_ZIP=ARCH/"NTX_3API_PAPER_CLOSURE_MASTER.zip"

make_zip(
    raw_stage,
    RAW_ZIP,
)

make_zip(
    RESULTS,
    RESULTS_ZIP,
)

make_zip(
    PAPER,
    PAPER_ZIP,
)

master_stage=BASE/"_master_export"
shutil.rmtree(
    master_stage,
    ignore_errors=True,
)
master_stage.mkdir()

for p in [
    RAW_ZIP,
    RESULTS_ZIP,
    PAPER_ZIP,
]:
    shutil.copy2(
        p,
        master_stage/p.name,
    )

copytree(
    RESULTS,
    master_stage/"results",
)

copytree(
    PAPER,
    master_stage/"paper_integration",
)

copytree(
    ENV,
    master_stage/"environment",
)

copytree(
    LOGS,
    master_stage/"logs",
)

make_zip(
    master_stage,
    MASTER_ZIP,
)

archive_rows=[]

for p in [
    RAW_ZIP,
    RESULTS_ZIP,
    PAPER_ZIP,
    MASTER_ZIP,
]:
    archive_rows.append({
        "file":p.name,
        "size_mib":round(
            p.stat().st_size/
            1024**2,
            3,
        ),
        "sha256":
            sha256_file(p),
    })

archive_df=pd.DataFrame(
    archive_rows
)

archive_df.to_csv(
    ARCH/
    "ARCHIVE_MANIFEST.csv",
    index=False,
)

display(archive_df)

In [ ]:
# ============================================================
# CELL 20 — FINAL STATUS + DOWNLOAD
# ============================================================
gate=claims.loc[
    claims.claim==
    "External-validation closure gate",
    "status",
].iloc[0]

final_status={
    "closure_gate":gate,
    "mode":MODE,
    "selftest":SELFTEST,
    "selected_routes":[
        {
            "family":x["family"],
            "provider":x["provider"],
            "model":x["model"],
        }
        for x in SELECTED
    ],
    "claims":
        claims.to_dict("records"),
    "master_zip":
        str(MASTER_ZIP),
    "master_sha256":
        sha256_file(MASTER_ZIP),
}

save_json(
    ARCH/
    "FINAL_CLOSURE_STATUS.json",
    final_status,
)

print(
    json.dumps(
        final_status,
        indent=2,
    )
)

try:
    from google.colab import files

    for p in [
        PAPER_ZIP,
        RESULTS_ZIP,
        RAW_ZIP,
        MASTER_ZIP,
    ]:
        files.download(
            str(p)
        )

except Exception as e:
    print(
        "Automatic browser download "
        "unavailable:",
        repr(e),
    )
    print(
        "Archives remain at:",
        ARCH,
    )